<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0; font-size: 2em;">🔍 Lab 04: Trace Your Hosted MAF Agent</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">
    See how hosted agent tracing differs — auto-instrumentation by the hosting adapter plus custom spans for your business logic
  </p>
</div>

**What We Learn:** How hosted agent tracing differs from prompted agent tracing — the hosting adapter provides auto-instrumentation for HTTP, model calls, and tool execution, plus you can add custom spans for your own logic.

---


## 🧳 The Contoso Travel Story

The Contoso Travel team has moved from prompted agents to the Microsoft Agent Framework (MAF), gaining the ability to run custom Python code inside tool functions — pricing calculations, data lookups, and complex business logic. But with greater power comes a new debugging challenge: when the agent runs custom code, you need to trace not just model calls but every step of your own logic.

- **The Problem:** When an agent runs custom Python code, you need to trace not just model calls but your own business logic — pricing calculations, data lookups, error handling paths. A wrong total on a flight quote could be a model hallucination *or* a bug in your pricing function, and without tracing you can't tell which.
- **This Lab Solves:** Understanding how MAF's hosting adapter auto-instruments HTTP/model/tool spans, and how to add custom spans for business logic so you can pinpoint issues in your own code alongside the auto-instrumented platform spans.

## 1. Setup

---


In [ ]:
# Load environment variables and enable GenAI tracing
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

# Enable GenAI tracing (must be set before importing tracing modules)
os.environ["AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING"] = "true"
os.environ["OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT"] = "true"

In [ ]:
# Connect to Microsoft Foundry project
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4.1-mini")

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print(f"✅ Connected to Microsoft Foundry")
print(f"   Model: {model_name}")

## 2. How Hosted Agent Tracing Differs

Tracing works differently depending on how your agent is built:

| Aspect | Prompted Agents | Hosted Agents (MAF) |
|---|---|---|
| **Instrumentation** | `AIProjectInstrumentor` instruments SDK calls | The hosting adapter (`from_agent_framework()`) auto-instruments HTTP requests, model calls, and tool invocations |
| **What you see** | SDK-level spans (responses.create, conversations.create) | Platform-level spans (HTTP, model inference, tool dispatch) |
| **Custom spans** | Optional — wrap your own calls with `tracer.start_as_current_span()` | Essential — add custom spans for your business logic (pricing, data lookups, validation) |
| **Depth** | Moderate — you see API calls | Deep — you see HTTP + model + tool + your custom code |

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2196f3; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>💡 Key Insight:</b> With prompted agents, <code>AIProjectInstrumentor</code> does the heavy lifting. With hosted agents, the hosting adapter provides auto-instrumentation for the platform layer, but <em>you</em> add custom spans for your own business logic using <code>tracer.start_as_current_span()</code>.
</div>

---


<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #533483; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 20px 0;">
  <h2 style="margin: 0;">Part A: Console Tracing</h2>
</div>

---


In [ ]:
# Configure OpenTelemetry console tracing
from azure.core.settings import settings
settings.tracing_implementation = "opentelemetry"

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter

# Setup console tracing
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("contoso-travel-maf")

print("✅ Console tracing configured")
print("   Traces will appear in the output below each cell")

## 3. Create and Trace a Hosted Agent

---


In [ ]:
# Create a Foundry agent for tracing demonstration
agent = project_client.agents.create_version(
    agent_name="contoso-travel-maf-traced",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions="You are the Contoso Travel Concierge. Help customers with travel questions about destinations, flights, hotels, and car rentals. Provide accurate pricing and availability.",
    ),
)
print(f"✅ Agent created: {agent.name} (v{agent.version})")

## 4. Run a Traced Travel Query

Watch the console output below — you'll see OpenTelemetry spans for each operation.

---


In [ ]:
# Run a traced travel query with a parent span
with tracer.start_as_current_span("contoso-travel-query"):
    conversation = openai_client.conversations.create()

    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        input="What are the best destinations for a summer vacation in Europe?",
    )

    print(f"\n🤖 Agent Response:\n{response.output_text}")

## 5. Observe Auto-Instrumented Spans

In the console output above, you should see trace spans including:

- **`contoso-travel-query`** — The parent span we created
  - **`conversations.create`** — Creating the conversation
  - **`responses.create`** — The model inference call

With a hosted MAF agent, the hosting adapter adds additional spans you wouldn't see with a prompted agent:

| Span | Source | What It Tells You |
|---|---|---|
| `HTTP POST` | Auto-instrumented | Network latency to the model endpoint |
| `chat.completions` | Auto-instrumented | Model inference time, token counts |
| `tool.<name>` | Auto-instrumented | Which tool was called, how long it took |
| `flight_search` | **Your custom span** | Your business logic execution time |


### Trace Span Hierarchy

When you view traces in the Foundry portal, the span tree for a hosted MAF agent looks like this:

```
contoso-travel-query                    ← your custom parent span
├── conversations.create                ← SDK auto-instrumented
└── responses.create                    ← SDK auto-instrumented
    ├── HTTP POST /chat/completions     ← hosting adapter auto-instrumented
    ├── chat gpt-4.1-mini               ← model inference span
    ├── tool.search_flights             ← hosting adapter auto-instrumented
    │   └── filter_by_price             ← YOUR custom business logic span
    └── chat gpt-4.1-mini               ← second model call with tool results
```

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #4caf50; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>✅ This is the key difference:</b> Hosted agents give you deeper visibility into the platform layer automatically. You focus on adding spans for <em>your</em> code.
</div>

---


## 6. Add Custom Spans for Business Logic

In a real MAF agent, your tool functions contain business logic. Here's how to trace it:

---


In [ ]:
# Simulate a traced tool function for flight search
def search_flights_traced(origin, destination, max_price=10000):
    """Search flights with full tracing of business logic."""
    with tracer.start_as_current_span("search_flights") as span:
        span.set_attribute("travel.origin", origin)
        span.set_attribute("travel.destination", destination)
        span.set_attribute("travel.max_price", max_price)

        # Simulate price filtering logic
        with tracer.start_as_current_span("filter_by_price") as filter_span:
            flights = [
                {"flight": "CT-FL-001", "price": 750, "class": "Economy"},
                {"flight": "CT-FL-002", "price": 1450, "class": "Business"},
                {"flight": "CT-FL-003", "price": 2800, "class": "First"},
            ]
            filtered = [f for f in flights if f["price"] <= max_price]
            filter_span.set_attribute("travel.results_total", len(flights))
            filter_span.set_attribute("travel.results_filtered", len(filtered))

        span.set_attribute("travel.results_count", len(filtered))
        return filtered


# Run the traced function
results = search_flights_traced("Seattle", "Paris", max_price=1500)
print(f"✈️ Found {len(results)} flights:")
for f in results:
    print(f"   {f['flight']} — ${f['price']} ({f['class']})")

print("\n📊 Check the console output above for nested spans:")
print("   search_flights → filter_by_price")

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #533483; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 20px 0;">
  <h2 style="margin: 0;">Part B: Azure Monitor Tracing</h2>
</div>

---


In [ ]:
# Clean up the console tracer to avoid duplicate spans
provider.shutdown()

from azure.monitor.opentelemetry import configure_azure_monitor

# Get the Application Insights connection string from the project
conn_string = project_client.telemetry.get_application_insights_connection_string()

# Configure Azure Monitor as the trace exporter
configure_azure_monitor(connection_string=conn_string)

# Get a new tracer
tracer = trace.get_tracer("contoso-travel-maf")

print("✅ Azure Monitor tracing configured")
print(f"   Traces will flow to Application Insights")
print(f"   View them in the Foundry portal → Tracing tab")

## 7. Run a Traced Travel Query (Azure Monitor)

This trace will appear in the Foundry portal. Note: it may take a few minutes for traces to appear in Azure Monitor.

---


In [ ]:
# Run a traced query that exports to Azure Monitor
with tracer.start_as_current_span("contoso-maf-paris-query") as span:
    span.set_attribute("travel.destination", "Paris")
    span.set_attribute("travel.query_type", "trip_planning")
    span.set_attribute("contoso.agent_name", agent.name)
    span.set_attribute("contoso.agent_type", "hosted-maf")

    conversation2 = openai_client.conversations.create()

    response = openai_client.responses.create(
        conversation=conversation2.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        input="I want to plan a week-long trip to Paris. What should I know about costs, weather, and must-see attractions?",
    )

    print(f"🤖 Agent Response:\n{response.output_text}")
    print(f"\n📊 Trace sent to Azure Monitor!")
    print(f"   Trace ID: {span.get_span_context().trace_id:032x}")

## 8. View in Foundry Portal

To view your traces:

1. Go to the [Microsoft Foundry portal](https://ai.azure.com)
2. Open your project
3. Click on the **Tracing** tab in the left navigation
4. You should see your traces listed with the span names we defined
5. Click on a trace to see the full span tree

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2196f3; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>💡 Compare with Lab 04 (Prompted Agents):</b> Hosted agent traces show more depth — you'll see HTTP-level spans from the hosting adapter alongside your custom business logic spans. Prompted agent traces only show SDK-level spans.
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #9c27b0; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>⏱️ Note:</b> Traces may take 2-5 minutes to appear in Azure Monitor after execution.
</div>


<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #4caf50; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>🔄 Local &amp; Deployed:</b> This tracing setup works identically whether you run the agent locally in a notebook or deploy it to Foundry Agent Service via <code>deploy.sh</code>. The hosting adapter auto-instruments both environments.
</div>

---

## 9. Cleanup

---


In [ ]:
# Clean up Foundry resources and close connections
openai_client.conversations.delete(conversation_id=conversation.id)
openai_client.conversations.delete(conversation_id=conversation2.id)
print("✅ Conversations deleted")

project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
print("✅ Agent deleted")

openai_client.close()
project_client.close()
credential.close()
print("✅ Clients closed")

## 10. Summary & Next Steps

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #2e8b57; padding: 18px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <h3 style="margin-top: 0; color: #1a1a1a;">✅ What We Accomplished</h3>
  <ul style="margin-bottom: 0;">
  <li>Understood how hosted agent tracing differs from prompted agent tracing</li>
  <li>Configured <b>console tracing</b> with <code>ConsoleSpanExporter</code> for local debugging</li>
  <li>Added <b>custom spans</b> for business logic using <code>tracer.start_as_current_span()</code></li>
  <li>Configured <b>Azure Monitor tracing</b> with <code>configure_azure_monitor</code> for production observability</li>
  <li>Ran traced travel queries and inspected auto-instrumented + custom spans</li>
  </ul>
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #1565c0; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <b>💡 Key Takeaway:</b> Hosted agents give you deeper tracing out of the box — the hosting adapter auto-instruments HTTP, model, and tool spans. Your job is to add custom spans for your own business logic so you can trace pricing calculations, data lookups, and error handling paths alongside the platform spans.
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #ff9800; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <b>➡️ Next:</b> In <a href="lab-05-evaluation.ipynb">Lab 05</a>, we'll <b>evaluate</b> the hosted MAF agent — and discover that evaluation is completely agent-agnostic!
</div>

---
